# Phase 0: Billboard Hot 100 Data Collection

**Goal**: Generate `songs.csv` - the shared contract with Billboard chart data from 1970-2023

In [3]:
import pandas as pd
import billboard
import time
from tqdm import tqdm
import re
from datetime import datetime

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
def normalize_artist_name(artist):
    """Normalize artist name for consistent matching"""
    if pd.isna(artist):
        return ""
    
    # Convert to lowercase
    artist = str(artist).lower()
    
    # Remove featuring artists
    artist = re.sub(r'\b(feat|ft|featuring)\.+', '', artist)
    artist = re.sub(r'\b(with|w/)\b.+', '', artist)
    
    # Remove extra whitespace and parentheses content
    artist = re.sub(r'\s+', ' ', artist)
    artist = re.sub(r'\([^)]*\)', '', artist)
    
    return artist.strip()

def normalize_title(title):
    """Normalize song title for consistent matching"""
    if pd.isna(title):
        return ""
    
    title = str(title).lower()
    
    # Remove extra info in parentheses
    title = re.sub(r'\([^)]*\)', '', title)
    
    # Remove special characters but keep spaces
    title = re.sub(r'[^a-z0-9\s]', '', title)
    
    return title.strip()

print("Normalization functions defined")

Normalization functions defined


In [ ]:
def collect_billboard_data(start_year=1970, end_year=2023):
    """Collect Billboard Hot 100 year-end charts for specified years"""
    
    all_songs = []
    
    for year in tqdm(range(start_year, end_year + 1), desc="Collecting Billboard data"):
        try:
            # Get year-end Hot 100 chart
            chart = billboard.ChartData('hot-100-songs', year=year)
            
            for entry in chart:
                song_data = {
                    'title': normalize_title(entry.title),
                    'artist': normalize_artist_name(entry.artist),
                    'year': year,
                    'chart_position': entry.rank,
                    'weeks_on_chart': 1  # Year-end charts don't have weeks data
                }
                all_songs.append(song_data)
            
            # Rate limiting
            time.sleep(1)
            
        except Exception as e:
            print(f"Error collecting data for {year}: {e}")
            continue
    
    return pd.DataFrame(all_songs)

print("Billboard collection function defined")

In [ ]:
# Collect the data
print("Starting Billboard data collection...")
df_songs = collect_billboard_data(1970, 2023)

print(f"\nCollected {len(df_songs)} songs total")
print(f"Years covered: {df_songs['year'].min()} - {df_songs['year'].max()}")

In [ ]:
# Remove duplicates (same song can appear in multiple years)
print(f"Before deduplication: {len(df_songs)} songs")

df_songs = df_songs.drop_duplicates(subset=['title', 'artist'], keep='first')

print(f"After deduplication: {len(df_songs)} unique songs")

# Add decade column
df_songs['decade'] = (df_songs['year'] // 10) * 10
df_songs['decade'] = df_songs['decade'].astype(str) + 's'

# Add top10 binary column
df_songs['top10'] = (df_songs['chart_position'] <= 10).astype(int)

print("\nSample of the data:")
print(df_songs.head())

In [ ]:
# Check distribution by decade
print("Songs by decade:")
print(df_songs['decade'].value_counts().sort_index())

print("\nTop 10 hits by decade:")
top10_by_decade = df_songs[df_songs['top10'] == 1]['decade'].value_counts().sort_index()
print(top10_by_decade)

In [ ]:
# Save to songs.csv
df_songs.to_csv('songs.csv', index=False)
print("Saved songs.csv successfully!")
print(f"Final dataset: {len(df_songs)} songs from {df_songs['year'].min()}-{df_songs['year'].max()}")

In [7]:
def collect_missing_years(start_year=1992, end_year=2006):
    """Collect mid-year weekly chart data for years missing from year-end charts"""
    all_songs = []
    
    for year in tqdm(range(start_year, end_year + 1), desc="Backfilling missing years"):
        try:
            date = f"{year}-06-30"
            chart = billboard.ChartData('hot-100', date=date)
            
            for entry in chart:
                all_songs.append({
                    'title': normalize_title(entry.title),
                    'artist': normalize_artist_name(entry.artist),
                    'year': year,
                    'chart_position': entry.rank,
                    'weeks_on_chart': 1
                })
            
            time.sleep(1)
        except Exception as e:
            print(f"Error for {year}: {e}")
            continue
    
    return pd.DataFrame(all_songs)

df_missing = collect_missing_years(1992, 2006)
df_existing = pd.read_csv('songs.csv')
df_combined = pd.concat([df_existing, df_missing], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['title', 'artist'], keep='first')
df_combined.to_csv('songs.csv', index=False)
print(f"Added {len(df_missing)} songs, total now: {len(df_combined)}")
print(df_combined.groupby((df_combined['year']//10)*10)['year'].count())


Backfilling missing years: 100%|██████████| 15/15 [06:46<00:00, 27.07s/it]

Added 1500 songs, total now: 4801
year
1970    796
1980    866
1990    984
2000    913
2010    888
2020    354
Name: year, dtype: int64
